# 07 · GAN condicional

Entrena una GAN condicional MLP sobre el bloque conjunto, con muestreo equilibrado por régimen y banda muerta en la actualización del discriminador.

**Responsable:** Oscar

**Entradas**

- `data/processed/ventanas.npz`

**Salidas**

- `models/generadores/cgan/ (modelo.pkl o .keras, historial.csv, meta.json)`
- `data/synthetic/cgan.npz`

**Tiempo estimado:** ~35 min en CPU (300 épocas; el juego adversario dobla el coste por paso).

**Independencia.** Este notebook solo lee `data/processed/ventanas.npz` (notebook 02) y solo escribe en `models/generadores/cgan/` y `data/synthetic/cgan.npz`. No depende de ningún otro notebook de generador ni de sus salidas, de modo que los notebooks 04 a 10 pueden ejecutarse en paralelo y en cualquier orden por distintas personas.

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import ventanas

part = ventanas.cargar_procesado()
train, val, test = part.train, part.val, part.test
print(train, val, test, sep="\n")

In [ ]:
from src import regimenes
from src.generadores import base

v = config.ventanas()
n_regimenes = config.n_regimenes()

bloque_train = ventanas.empaquetar(train)
print("bloque de train:", bloque_train.shape, "· d esperada:", ventanas.dimension_bloque(v))
regimenes.distribucion(train.y_reg, n_regimenes)

## Por qué una GAN condicional y cómo se lee su convergencia

La GAN no optimiza una verosimilitud: juega. No converge a un mínimo sino a un
punto de silla, y por eso su curva de pérdida **no baja**. Leerla como una loss
normal lleva a conclusiones falsas en las dos direcciones: una pérdida del
generador que baja mucho suele significar que el discriminador ha dejado de
discriminar, y una que sube no implica divergencia.

La referencia es log 2 ≈ 0.693. Ahí es donde deben oscilar ambas pérdidas si el
juego está equilibrado: es el valor de la entropía cruzada cuando el discriminador
asigna 0.5 a todo, es decir, cuando no distingue.

Tres decisiones de diseño que hacen que el entrenamiento sobreviva:

**Suavizado de etiquetas por un lado.** La etiqueta de "real" es 0.9 en vez de 1.0.
Impide que el discriminador empuje sus logits a infinito y mate el gradiente hacia
el generador.

**Banda muerta en el discriminador.** Si su precisión media móvil supera 0.80, se
salta su actualización esa iteración. Es la versión sensata del `ratio` adaptativo
del cuaderno de clase, que modulaba el tamaño de lote —rompiendo BatchNorm y el
paso efectivo de Adam— en vez de la frecuencia de actualización, que es el grado de
libertad `k` del algoritmo original.

**Lotes equilibrados por régimen.** Con la proporción real (~10 % de crisis) un
lote de 128 trae unas 13 ventanas de crisis y el generador apenas aprende
`p(x | crisis)`, que es justo lo que el taller quiere sobre-representar. Las
etiquetas de las muestras falsas se toman del mismo vector que las reales, de modo
que la marginal de `y` es idéntica a ambos lados y el discriminador no puede usar
la frecuencia de clase como atajo.

`beta_1 = 0.5` en Adam en lugar del 0.9 por defecto: reducir el momento de primer
orden amortigua las oscilaciones del juego.

In [ ]:
generador = base.instanciar(
    "cgan",
    n_regimenes=n_regimenes,
    dim_latente=128,
    epocas=300,
    tam_lote=128,
    tasa_aprendizaje=2e-4,
    suavizado=0.9,
    umbral_disc=0.80,
    equilibrar_clases=True,
    hilos_torch=4,
    verboso=True,
)
generador.fit(bloque_train, train.y_reg)
generador

## Convergencia

La línea horizontal marca el equilibrio log 2 = 0.693. Sin ella estas curvas son
prácticamente ilegibles, porque no bajan monótonamente y no hay forma de juzgar a
ojo si el juego se sostuvo.

Lo que hay que ver: ambas pérdidas oscilando alrededor de la referencia, sin que
ninguna se dispare ni se hunda; las precisiones del discriminador sobre reales y
falsos cerca de 0.5; y `fraccion_pasos_disc` claramente por debajo de 1 en la
segunda mitad, señal de que la banda muerta está actuando y el discriminador no ha
ganado el juego.

El fallo típico —colapso de modo— se ve mejor en la figura PCA de la celda
siguiente que en estas curvas.

In [ ]:
fig, eje = plt.subplots(figsize=(10, 5))
viz.curva_convergencia(
    generador.historial,
    "Convergencia · " + generador.etiqueta,
    referencia=0.693,
    eje=eje,
)
viz.guardar(fig, "convergencia_cgan")

generador.historial.tail(5).round(3)

## Inspección visual

Proyección PCA de reales y sintéticos, con la PCA ajustada **solo con los reales**
para que los ejes describan la estructura del mercado y no la del generador.

Es la comprobación más rápida y la que detecta los dos fallos gruesos: si la nube
sintética no cubre la real, el generador ha colapsado a un modo; si la desborda
ampliamente, está inventando configuraciones de mercado que nunca ocurrieron.

Se mira el régimen de crisis porque es el que tiene menos datos reales y, por
tanto, donde el generador tiene más margen para desviarse.

In [ ]:
CRISIS = n_regimenes - 1

muestra_crisis = generador.generate(600, regimen=CRISIS)
reales_crisis = bloque_train[train.y_reg == CRISIS]

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
viz.real_vs_sintetico(bloque_train, generador.generate(600, regimen=0),
                      "{} · régimen de calma".format(generador.etiqueta), eje=ejes[0])
viz.real_vs_sintetico(reales_crisis, muestra_crisis,
                      "{} · régimen de crisis".format(generador.etiqueta), eje=ejes[1])
fig.tight_layout()
viz.guardar(fig, "pca_" + generador.nombre)

print("reales de crisis:", len(reales_crisis), "· sintéticos generados:", len(muestra_crisis))

## Banco de muestras

Se genera un banco uniforme por régimen y se exporta a `data/synthetic/`. La mezcla
concreta de cada dataset la decide el notebook 11 muestreando de este banco, no
volviendo a invocar al generador: así el barrido no necesita tener los siete
modelos cargados en memoria y dos ejecuciones del notebook 12 usan exactamente las
mismas muestras sintéticas.

El banco es uniforme —no replica el desbalance real— porque la política de reparto
es un grado de libertad del experimento y se aplica después.

In [ ]:
MUESTRAS_POR_REGIMEN = 3000

reparto = {k: MUESTRAS_POR_REGIMEN for k in range(n_regimenes)}
bloques_sint, y_sint = generador.generate_dataset(reparto)

print("banco:", bloques_sint.shape, "· etiquetas:", np.bincount(y_sint, minlength=n_regimenes))
print("rango de valores:", round(float(bloques_sint.min()), 2), "→",
      round(float(bloques_sint.max()), 2),
      "(referencia real:", round(float(bloque_train.min()), 2), "→",
      round(float(bloque_train.max()), 2), ")")

## Persistencia

`guardar()` deja el modelo, la curva de convergencia y los metadatos en
`models/generadores/`. Es lo que permite que el resto del grupo salte directamente
al análisis sin reentrenar nada.

In [ ]:
ruta_muestras = generador.exportar_muestras(bloques_sint, y_sint)
ruta_modelo = generador.guardar()

print("muestras:", ruta_muestras)
print("modelo:  ", ruta_modelo)
pd.Series(generador.resumen_convergencia()).round(4)

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_MODELOS_GEN / "cgan" / "meta.json",
    src.DIR_MODELOS_GEN / "cgan" / "historial.csv",
    src.DIR_SINTETICO / "cgan.npz",
    src.DIR_FIGURAS / "convergencia_cgan.png",
    src.DIR_FIGURAS / "pca_cgan.png",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
